# NEXORA Phase 5 — Decision Intelligence Engine Analysis

Analysis and presentation only -- **the production pipeline does not depend on this notebook**. `python -m decision_engine.generate_decisions` and `python -m etl.validate_decision_engine` are fully self-contained. See `docs/decision_engine.md` for the full design writeup.

- Decision distribution
- Severity distribution
- Decisions by business domain
- Revenue at risk
- Project exposure
- Customer exposure
- Example decision cards

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from mining.common import fetch_dataframe

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

decisions = fetch_dataframe("SELECT * FROM ANALYTICS.DI_DECISIONS")
for c in ["PRIORITY_SCORE", "BUSINESS_IMPACT_VALUE", "CONFIDENCE_SCORE"]:
    decisions[c] = decisions[c].astype(float)
summary = fetch_dataframe("SELECT * FROM ANALYTICS.DI_EXECUTIVE_SUMMARY")
print(decisions.shape)
summary

## Decision distribution by type

In [ ]:
decisions["DECISION_TYPE"].value_counts().plot(kind="barh", figsize=(7, 4))
plt.title("Decisions by type"); plt.xlabel("count"); plt.tight_layout(); plt.show()

## Severity distribution

In [ ]:
order = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
colors = {"CRITICAL": "#b91c1c", "HIGH": "#ea580c", "MEDIUM": "#ca8a04", "LOW": "#16a34a"}
counts = decisions["SEVERITY"].value_counts().reindex(order).fillna(0)
counts.plot(kind="bar", color=[colors[s] for s in order], figsize=(6, 4))
plt.title("Severity distribution"); plt.ylabel("count"); plt.xticks(rotation=0); plt.show()

## Decisions by business domain x severity

In [ ]:
pivot = pd.crosstab(decisions["DECISION_TYPE"], decisions["SEVERITY"])[["CRITICAL", "HIGH", "MEDIUM", "LOW"]]
pivot.plot(kind="barh", stacked=True, figsize=(8, 5), color=[colors[s] for s in order])
plt.title("Decisions by domain and severity"); plt.xlabel("count"); plt.tight_layout(); plt.show()
pivot

## Priority score distribution

In [ ]:
decisions["PRIORITY_SCORE"].hist(bins=30, figsize=(7, 4))
plt.axvline(80, color="red", linestyle="--", label="CRITICAL >= 80")
plt.axvline(60, color="orange", linestyle="--", label="HIGH >= 60")
plt.axvline(40, color="gold", linestyle="--", label="MEDIUM >= 40")
plt.legend(); plt.title("Priority score distribution"); plt.xlabel("priority_score"); plt.show()

## Revenue at risk / opportunity

In [ ]:
print(f"Revenue at risk (distinct customers): ${summary['REVENUE_AT_RISK'].iloc[0]:,.0f}")
print(f"Payment value at risk (distinct invoices): ${summary['PAYMENT_VALUE_AT_RISK'].iloc[0]:,.0f}")
print(f"Revenue opportunity value: ${summary['REVENUE_OPPORTUNITY_VALUE'].iloc[0]:,.0f}")

## Project exposure

In [ ]:
project_decisions = decisions[decisions["ENTITY_TYPE"] == "PROJECT"]
print(f"Projects at risk: {project_decisions['ENTITY_ID'].nunique()}")
print(f"Total budget exposure: ${project_decisions['BUSINESS_IMPACT_VALUE'].sum():,.0f}")
project_decisions.nlargest(5, "PRIORITY_SCORE")[["ENTITY_ID", "TITLE", "PRIORITY_SCORE", "SEVERITY", "BUSINESS_IMPACT_VALUE"]]

## Customer exposure

In [ ]:
customer_decisions_df = decisions[decisions["ENTITY_TYPE"] == "CUSTOMER"]
print(f"Customers at risk: {customer_decisions_df['ENTITY_ID'].nunique()}")
print(f"Total ARR exposure: ${customer_decisions_df['BUSINESS_IMPACT_VALUE'].sum():,.0f}")
customer_decisions_df.nlargest(5, "PRIORITY_SCORE")[["ENTITY_ID", "TITLE", "PRIORITY_SCORE", "SEVERITY", "BUSINESS_IMPACT_VALUE", "CONFIDENCE_SCORE"]]

## Example decision cards -- full detail, one per domain

In [ ]:
def show_card(decision_type):
    row = decisions[decisions["DECISION_TYPE"] == decision_type].sort_values("PRIORITY_SCORE", ascending=False).iloc[0]
    print("=" * 90)
    print(f"{row['TITLE']}  [{row['SEVERITY']} | priority {row['PRIORITY_SCORE']:.1f} | confidence {row['CONFIDENCE_SCORE']:.1f}]")
    print("-" * 90)
    print("WHAT HAPPENED:", row["WHAT_HAPPENED"])
    print("WHY:", row["WHY_IT_HAPPENED"])
    print("PREDICTED OUTCOME:", row["PREDICTED_OUTCOME"])
    print(f"BUSINESS IMPACT: {row['BUSINESS_IMPACT_TYPE']} = ${row['BUSINESS_IMPACT_VALUE']:,.0f}")
    print("RECOMMENDED ACTIONS:")
    for i in range(1, 6):
        action = row.get(f"RECOMMENDED_ACTION_{i}")
        if action:
            print(f"  {i}. {action}")
    print()

for dtype in ["CUSTOMER_RETENTION", "PROJECT_DELIVERY", "PAYMENT_COLLECTION", "REVENUE_OPPORTUNITY", "BUSINESS_ANOMALY"]:
    show_card(dtype)